# 03 — Pipeline HÍBRIDO (todo junto)

**El pipeline completo hasta ahora, en uno:**
```
YOLO (cada frame) -> cajas -> SAM3 box-prompt -> máscaras tight
   -> tracker IoU (IDs estables, color por track, coasting en oclusiones)
   -> green_floor por SAM3 text-prompt (cada N frames)
```
Junta: re-detección por frame (YOLO) + máscaras finas (SAM3 = centro) + consistencia temporal e IDs (tracker). Corre en GPU del pod.

Lógica en `hybrid.py`. Ver `context.md`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))
import pipeline_yolo_sam3 as pl
import hybrid as hb

REPO = Path('/workspace/FutBotMX-UAQTeam')
F2 = Path.cwd()
models = pl.load_models(REPO/'assets'/'sam3', F2/'best.pt', device='cuda')

In [ ]:
VIDEO = REPO / 'data/raw/17Abril/Cámaras/IMG_9871.MOV'   # testing, no visto
hb.render_hybrid(VIDEO, F2/'demo_hybrid_IMG_9871.mp4', models,
                 conf=0.4, green_every=10, iou_thr=0.3, max_age=8)

## Parámetros
- `green_every`: cada cuántos frames recalcula green_floor (estático, lento).
- `iou_thr`: umbral IoU para ligar detección↔track (ID estable).
- `max_age`: frames que un track sobrevive sin detección (coasting en oclusión).

## Siguiente del pipeline
Homografía (con green_floor), eventos (gol con yellow_zone + velocidad balón), LoRA team-aware (robot_a/robot_b).